# 02 — 전처리 (Preprocessing)
### Tennessee Eastman Process 이상탐지 프로젝트

이 노트북에서 하는 것:
1. simulationRun 기준 train / test 분리 (데이터 누수 방지)
2. 동적 피처 생성 (rolling_mean · rolling_std · diff · z-score)
3. StandardScaler — train fit, test transform only
4. 전처리 결과 저장 → `data/processed/`
5. 다음 단계 (Isolation Forest · XGBoost) 입력 형식 확인



## 0. 환경 설정

In [1]:
import sys, os

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

print('환경 설정 완료')


환경 설정 완료


## 1. 데이터 로딩

In [2]:
from src.data_loader import (
    load_tep, FEATURE_COLS, XMEAS_COLS, XMV_COLS, FAULT_DESC
)

FAULT_TYPES = [1, 2, 4, 5]

train_df, test_df = load_tep(
    data_dir='../data/raw',
    fault_types=FAULT_TYPES,
    verbose=True
)

print(f'train shape : {train_df.shape}')
print(f'test  shape : {test_df.shape}')
print(f'simulationRun 고유값 (train): {sorted(train_df["simulationRun"].unique())}')


  로딩 완료: TEP_FaultFree_Training.csv  (250000, 55)
  로딩 완료: TEP_FaultFree_Testing.csv  (480000, 55)
  로딩 완료: TEP_Faulty_Training.csv  (5000000, 55)
  로딩 완료: TEP_Faulty_Testing.csv  (9600000, 55)

TEP 데이터 로딩 완료

[학습]  총 1,250,000행  |  정상: 250,000  |  이상: 1,000,000  (80.0%)

[테스트]  총 2,400,000행  |  정상: 480,000  |  이상: 1,920,000  (80.0%)

사용한 이상 유형: [1, 2, 4, 5]
이상 유형 설명:
  fault  1: A/C 피드 비율 이상 (Step)
  fault  2: B 성분 조성 이상 (Step)
  fault  4: 반응기 냉각수 입구온도 이상 (Step)
  fault  5: 냉각기 냉각수 입구온도 이상 (Step)
train shape : (1250000, 56)
test  shape : (2400000, 56)
simulationRun 고유값 (train): [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0, 54.0, 55.0, 56.0, 57.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0, 70.0, 71

## 2. simulationRun 기준 Train / Validation 분리

> **왜 simulationRun 단위로 분리하는가?**  
> 같은 시뮬레이션 런 내 시점들은 시계열 자기상관이 있음.  
> 무작위 행 분할 시 미래 정보가 학습에 유입되는 **데이터 누수** 발생.  
> 런 단위 분할로 완전히 독립적인 검증셋을 확보함.


In [3]:
# ── simulationRun 단위 GroupShuffleSplit ──
# test_df는 이미 별도로 존재하므로,
# train_df를 train / validation 으로 나눔.

VAL_RATIO   = 0.2     # validation 비율
RANDOM_SEED = 42

runs = train_df['simulationRun'].values

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_RATIO, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(train_df, groups=runs))

tr = train_df.iloc[train_idx].copy().reset_index(drop=True)
va = train_df.iloc[val_idx].copy().reset_index(drop=True)
te = test_df.copy().reset_index(drop=True)

print('── 분리 결과 ──')
print(f'  Train      : {len(tr):>9,}행  | runs: {tr["simulationRun"].nunique()}개')
print(f'  Validation : {len(va):>9,}행  | runs: {va["simulationRun"].nunique()}개')
print(f'  Test       : {len(te):>9,}행  | runs: {te["simulationRun"].nunique()}개')
print()

# train/validation은 같은 training 파일에서 나누므로 simulationRun 겹침이 없어야 함
overlap_tv = set(tr['simulationRun']) & set(va['simulationRun'])
print(f'  train-val 런 겹침  : {len(overlap_tv)}  (0이어야 정상)')
assert len(overlap_tv) == 0, '데이터 누수! train-val simulationRun 겹침 발생'

# test_df는 원본 testing 파일에서 따로 제공되므로,
# simulationRun 번호가 train과 겹쳐도 같은 실험 샘플이 섞인 것은 아님
overlap_tt = set(tr['simulationRun']) & set(te['simulationRun'])
print(f'  train-test 런 번호 겹침 : {len(overlap_tt)}  (TEP 원본 구조상 허용)')

print('\n train/validation 기준 데이터 누수 없음 확인')


── 분리 결과 ──
  Train      : 1,000,000행  | runs: 400개
  Validation :   250,000행  | runs: 100개
  Test       : 2,400,000행  | runs: 500개

  train-val 런 겹침  : 0  (0이어야 정상)
  train-test 런 번호 겹침 : 400  (TEP 원본 구조상 허용)

 train/validation 기준 데이터 누수 없음 확인


## 3. 동적 피처 생성 (Rolling · Diff · Z-score)

EDA에서 확인한 결과:  
- **xmeas_9, xmeas_17** — 평균값 변화 작음 → rolling_std 필수  
- **xmeas_10, xmv_6, xmeas_34, xmeas_28** — 민감도 상위 변수

동적 피처는 **simulationRun 경계 내에서만** 계산 (경계 혼합 방지).


In [4]:
# ── 동적 피처 대상 변수 ──
# 민감도 상위 6개 + 평균 변화 작은 2개
DYNAMIC_VARS = [
    'xmeas_10', 'xmv_6', 'xmeas_34', 'xmeas_28',
    'xmeas_1',  'xmv_3',
    'xmeas_9',  'xmeas_17',   # 평균 변화는 작지만 동적 변동성 확인용 후보
]
DYNAMIC_VARS = [v for v in DYNAMIC_VARS if v in FEATURE_COLS]

WINDOW = 20   # 타임스텝 단위 (EDA와 동일)

def add_dynamic_features(df: pd.DataFrame,
                          vars_list: list,
                          window: int = 20,
                          zscore_stats: dict = None) -> pd.DataFrame:
    """
    Parameters
    ----------
    df           : 원본 데이터프레임 (simulationRun 컬럼 필요)
    vars_list    : 동적 피처를 만들 변수 목록
    window       : rolling 윈도우 크기
    zscore_stats : {'mean': Series, 'std': Series}
                   None이면 df 자체의 정상 데이터로 계산 (train 전용)
                   train 외 split에는 반드시 train 통계 전달

    Returns
    -------
    df에 동적 컬럼이 추가된 데이터프레임
    """
    df = df.copy()

    for var in vars_list:
        grp = df.groupby(['faultNumber', 'simulationRun'])[var]

        # rolling mean / std  (simulationRun 경계 내)
        df[f'{var}_rmean'] = grp.transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
        df[f'{var}_rstd'] = grp.transform(
            lambda x: x.rolling(window, min_periods=1).std().fillna(0)
        )

        # 1-step diff  (simulationRun 경계에서 NaN → 0 처리)
        df[f'{var}_diff'] = grp.transform(lambda x: x.diff().fillna(0))

        # z-score (정상 데이터 기준 μ·σ 사용)
        if zscore_stats is not None:
            mu  = zscore_stats['mean'][var]
            sig = zscore_stats['std'][var] + 1e-8
        else:
            # train split: 정상 샘플만으로 통계 계산
            normal_mask = df['label'] == 0
            mu  = df.loc[normal_mask, var].mean()
            sig = df.loc[normal_mask, var].std() + 1e-8

        df[f'{var}_zscore'] = (df[var] - mu) / sig

    return df

# ── Train 기준 z-score 통계 계산 ──
normal_tr = tr[tr['label'] == 0]
zscore_stats = {
    'mean': normal_tr[DYNAMIC_VARS].mean(),
    'std' : normal_tr[DYNAMIC_VARS].std(),
}

# ── 동적 피처 추가 ──
print('동적 피처 생성 중...')
tr_feat = add_dynamic_features(tr, DYNAMIC_VARS, WINDOW, zscore_stats)
va_feat = add_dynamic_features(va, DYNAMIC_VARS, WINDOW, zscore_stats)
te_feat = add_dynamic_features(te, DYNAMIC_VARS, WINDOW, zscore_stats)

DYNAMIC_COLS = [c for c in tr_feat.columns
                if any(c.endswith(s) for s in ['_rmean','_rstd','_diff','_zscore'])]

print(f'  생성된 동적 피처 수 : {len(DYNAMIC_COLS)}개')
print(f'  train  shape : {tr_feat.shape}')
print(f'  val    shape : {va_feat.shape}')
print(f'  test   shape : {te_feat.shape}')
print(f'\n  동적 피처 목록 (앞 8개):')
for c in DYNAMIC_COLS[:8]:
    print(f'    {c}')


동적 피처 생성 중...
  생성된 동적 피처 수 : 32개
  train  shape : (1000000, 88)
  val    shape : (250000, 88)
  test   shape : (2400000, 88)

  동적 피처 목록 (앞 8개):
    xmeas_10_rmean
    xmeas_10_rstd
    xmeas_10_diff
    xmeas_10_zscore
    xmv_6_rmean
    xmv_6_rstd
    xmv_6_diff
    xmv_6_zscore


## 4. 결측값 · 무한대 처리

In [5]:
ALL_FEATURE_COLS = FEATURE_COLS + DYNAMIC_COLS

for split_name, df in [('train', tr_feat), ('val', va_feat), ('test', te_feat)]:
    inf_cnt = np.isinf(df[ALL_FEATURE_COLS].values).sum()
    nan_cnt = df[ALL_FEATURE_COLS].isnull().sum().sum()
    print(f'{split_name:6s} — inf: {inf_cnt}, nan: {nan_cnt}')

# 안전하게 inf → NaN → 0 처리
for df in [tr_feat, va_feat, te_feat]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df[ALL_FEATURE_COLS] = df[ALL_FEATURE_COLS].fillna(0)

print('\n 결측/무한대 처리 완료')


train  — inf: 0, nan: 0
val    — inf: 0, nan: 0
test   — inf: 0, nan: 0

 결측/무한대 처리 완료


## 5. StandardScaler 적용

**규칙:** train split에서만 `fit` → val·test에는 `transform` 만 적용  
(scaler는 `models/scaler.pkl`에 저장해 추론 시 재사용)


In [6]:
scaler = StandardScaler()

# fit — train만
X_tr = scaler.fit_transform(tr_feat[ALL_FEATURE_COLS])

# transform — val, test
X_va = scaler.transform(va_feat[ALL_FEATURE_COLS])
X_te = scaler.transform(te_feat[ALL_FEATURE_COLS])

y_tr = tr_feat['label'].values
y_va = va_feat['label'].values
y_te = te_feat['label'].values

group_tr = tr_feat['simulationRun'].values   # 평가용 group 정보
group_va = va_feat['simulationRun'].values
group_te = te_feat['simulationRun'].values

print(f'X_tr : {X_tr.shape}  (피처 수 = {len(ALL_FEATURE_COLS)})')
print(f'X_va : {X_va.shape}')
print(f'X_te : {X_te.shape}')
print()
print(f'y_tr 분포 — 정상: {(y_tr==0).sum():,}  이상: {(y_tr==1).sum():,}')
print(f'y_va 분포 — 정상: {(y_va==0).sum():,}  이상: {(y_va==1).sum():,}')
print()

# scaler 저장
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
print('✅ scaler 저장: models/scaler.pkl')


X_tr : (1000000, 84)  (피처 수 = 84)
X_va : (250000, 84)
X_te : (2400000, 84)

y_tr 분포 — 정상: 200,000  이상: 800,000
y_va 분포 — 정상: 50,000  이상: 200,000

✅ scaler 저장: models/scaler.pkl


## 6. 전처리 결과 저장

In [7]:
# numpy 배열로 저장 (모델 노트북에서 바로 로드 가능)
np.save('../data/processed/X_tr.npy', X_tr)
np.save('../data/processed/X_va.npy', X_va)
np.save('../data/processed/X_te.npy', X_te)
np.save('../data/processed/y_tr.npy', y_tr)
np.save('../data/processed/y_va.npy', y_va)
np.save('../data/processed/y_te.npy', y_te)
np.save('../data/processed/group_tr.npy', group_tr)
np.save('../data/processed/group_va.npy', group_va)
np.save('../data/processed/group_te.npy', group_te)

# 피처 이름 목록도 저장 (모델 해석 시 필요)
feature_names = pd.Series(ALL_FEATURE_COLS)
feature_names.to_csv('../data/processed/feature_names.csv', index=False, header=False)

print('저장 완료:')
for f in ['X_tr','X_va','X_te','y_tr','y_va','y_te','group_tr','group_va','group_te']:
    path = f'../data/processed/{f}.npy'
    size = os.path.getsize(path) / 1e6
    print(f'  {path}  ({size:.1f} MB)')
print(f'  ../data/processed/feature_names.csv')


저장 완료:
  ../data/processed/X_tr.npy  (672.0 MB)
  ../data/processed/X_va.npy  (168.0 MB)
  ../data/processed/X_te.npy  (1612.8 MB)
  ../data/processed/y_tr.npy  (8.0 MB)
  ../data/processed/y_va.npy  (2.0 MB)
  ../data/processed/y_te.npy  (19.2 MB)
  ../data/processed/group_tr.npy  (8.0 MB)
  ../data/processed/group_va.npy  (2.0 MB)
  ../data/processed/group_te.npy  (19.2 MB)
  ../data/processed/feature_names.csv


## 7. 저장 결과 검증

In [8]:
print('=== 저장 파일 로드 검증 ===')
X_tr_loaded = np.load('../data/processed/X_tr.npy')
y_tr_loaded = np.load('../data/processed/y_tr.npy')
feat_names  = pd.read_csv('../data/processed/feature_names.csv', header=None)[0].tolist()

print(f'X_tr_loaded : {X_tr_loaded.shape}')
print(f'y_tr_loaded : {y_tr_loaded.shape}')
print(f'feature 수  : {len(feat_names)}개')
print()

# 스케일 확인 — 평균 ≈ 0, std ≈ 1 이어야 함
means = X_tr_loaded.mean(axis=0)
stds  = X_tr_loaded.std(axis=0)
print(f'X_tr 평균 범위 : [{means.min():.4f}, {means.max():.4f}]  (≈ 0 이어야 함)')
print(f'X_tr std  범위 : [{stds.min():.4f}, {stds.max():.4f}]  (대부분 ≈ 1, 상수 피처는 0 가능)')
print()
print('✅ 전처리 파이프라인 검증 완료')


=== 저장 파일 로드 검증 ===
X_tr_loaded : (1000000, 84)
y_tr_loaded : (1000000,)
feature 수  : 84개

X_tr 평균 범위 : [-0.0000, 0.0000]  (≈ 0 이어야 함)
X_tr std  범위 : [1.0000, 1.0000]  (대부분 ≈ 1, 상수 피처는 0 가능)

✅ 전처리 파이프라인 검증 완료


In [9]:
print(f'''
╔══════════════════════════════════════════════════════════════╗
║          다음 노트북 입력 형식 가이드                        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  # 공통 로딩 코드                                            ║
║  # 03_isolation_forest.ipynb / 04_xgboost_baseline.ipynb     ║
║                                                              ║
║  import numpy as np, pandas as pd                            ║
║                                                              ║
║  X_tr = np.load("../data/processed/X_tr.npy")                ║
║  X_va = np.load("../data/processed/X_va.npy")                ║
║  X_te = np.load("../data/processed/X_te.npy")                ║
║  y_tr = np.load("../data/processed/y_tr.npy")                ║
║  y_va = np.load("../data/processed/y_va.npy")                ║
║  y_te = np.load("../data/processed/y_te.npy")                ║
║                                                              ║
║  group_tr = np.load("../data/processed/group_tr.npy")        ║
║  group_va = np.load("../data/processed/group_va.npy")        ║
║  group_te = np.load("../data/processed/group_te.npy")        ║
║                                                              ║
║  feat = pd.read_csv("../data/processed/feature_names.csv",   ║
║                     header=None)[0].tolist()                 ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  03_isolation_forest.ipynb                                   ║
║                                                              ║
║    from sklearn.ensemble import IsolationForest              ║
║                                                              ║
║    X_tr_normal = X_tr[y_tr == 0]                             ║
║                                                              ║
║    model = IsolationForest(                                  ║
║        contamination=0.05,                                   ║
║        n_estimators=200,                                     ║
║        random_state=42,                                      ║
║        n_jobs=-1                                             ║
║    )                                                         ║
║                                                              ║
║    model.fit(X_tr_normal)                                    ║
║                                                              ║
║    → 정상 데이터만 학습하는 비지도 이상탐지 baseline          ║
║    → 실제 공정에서는 이상 데이터가 희소하므로 현장 논리와 맞음 ║
║    → AUROC / PR-AUC / Recall / Detection Delay 기준 평가     ║
║                                                              ║
║  04_xgboost_baseline.ipynb                                   ║
║                                                              ║
║    from xgboost import XGBClassifier                         ║
║                                                              ║
║    scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()  ║
║                                                              ║
║    model = XGBClassifier(                                    ║
║        n_estimators=500,                                     ║
║        learning_rate=0.05,                                   ║
║        max_depth=6,                                          ║
║        subsample=0.8,                                        ║
║        colsample_bytree=0.8,                                 ║
║        scale_pos_weight=scale_pos_weight,                    ║
║        random_state=42,                                      ║
║        eval_metric="logloss"                                 ║
║    )                                                         ║
║                                                              ║
║    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=50) ║
║                                                              ║
║    → 지도학습 baseline                                       ║
║    → scale_pos_weight로 클래스 불균형 보정                   ║
║    → feature_importances_로 원본/동적 피처 기여도 확인        ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  피처 수 요약                                                ║
║    원본 정적 피처  : 52개                                    ║
║    추가 동적 피처  : {len(DYNAMIC_COLS)}개                    ║
║    총 입력 피처    : {len(ALL_FEATURE_COLS)}개                ║
╚══════════════════════════════════════════════════════════════╝
''')


╔══════════════════════════════════════════════════════════════╗
║          다음 노트북 입력 형식 가이드                        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  # 공통 로딩 코드                                            ║
║  # 03_isolation_forest.ipynb / 04_xgboost_baseline.ipynb     ║
║                                                              ║
║  import numpy as np, pandas as pd                            ║
║                                                              ║
║  X_tr = np.load("../data/processed/X_tr.npy")                ║
║  X_va = np.load("../data/processed/X_va.npy")                ║
║  X_te = np.load("../data/processed/X_te.npy")                ║
║  y_tr = np.load("../data/processed/y_tr.npy")                ║
║  y_va = np.load("../data/processed/y_va.npy")                ║
║  y_te = np.load("../data/processed/y_te.npy")                ║
║                                         